<a href="https://colab.research.google.com/github/pranjalgupta29/momentum_strategy_quant/blob/main/task_submission_HFT_Talent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!ls
!pip install ta
!pip install backtesting

sample_data
  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=d233082f0a512f3b23849ab03b85c78ef3b4f2c23f41880327ee02b9e685569e
  Stored in directory: /root/.cache/pip/wheels/a1/d7/29/7781cc5eb9a3659d032d7d15bdd0f49d07d2b24fec29f44bc4
Successfully built ta
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.4/191.4 kB 3.2 MB/s eta 0:00:00


In [15]:
# Import libraries
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.metrics import accuracy_score
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.trend import MACD, SMAIndicator
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import OnBalanceVolumeIndicator
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
from backtesting.test import GOOG
import warnings
warnings.filterwarnings('ignore')


In [16]:
# === Step 1: Feature Engineering === #
def add_features(df):
    df['RSI'] = RSIIndicator(close=df['Close'], window=14).rsi()
    df['MACD'] = MACD(close=df['Close']).macd_diff()
    df['STOCH_K'] = StochasticOscillator(df['High'], df['Low'], df['Close']).stoch()
    df['ATR'] = AverageTrueRange(df['High'], df['Low'], df['Close']).average_true_range()
    bb = BollingerBands(df['Close'])
    df['BB_width'] = bb.bollinger_hband() - bb.bollinger_lband()
    df['SMA50'] = SMAIndicator(df['Close'], window=50).sma_indicator()
    df['SMA200'] = SMAIndicator(df['Close'], window=200).sma_indicator()
    df['OBV'] = OnBalanceVolumeIndicator(df['Close'], df['Volume']).on_balance_volume()
    df['Return_5d'] = df['Close'].pct_change(5)
    df['Sharpe_20'] = df['Close'].pct_change().rolling(20).mean() / df['Close'].pct_change().rolling(20).std()
    df = df.dropna()
    return df

In [17]:
# Hyper parameters unoptimized
def train_model(df):
    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    features = ['RSI', 'MACD', 'STOCH_K', 'ATR', 'BB_width',
                'SMA50', 'SMA200', 'OBV', 'Return_5d', 'Sharpe_20']
    X = df[features]
    y = df['Target']
    X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.3)
    model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    model.fit(X_train, y_train)
    df['pred_proba'] = model.predict_proba(X)[:,1]
    return df, model

In [18]:
# With Hyper parameters tuned
def train_model_tuning(df):
    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    features = ['RSI', 'MACD', 'STOCH_K', 'ATR', 'BB_width',
                'SMA50', 'SMA200', 'OBV', 'Return_5d', 'Sharpe_20']
    X = df[features]
    y = df['Target']
    X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.3)

    param_grid = {
        'n_estimators': [50, 100],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1],
        'subsample': [0.8, 1.0]
    }

    xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    tscv = TimeSeriesSplit(n_splits=5)

    grid_search = GridSearchCV(xgb_clf, param_grid, cv=tscv, scoring='accuracy', verbose=1, n_jobs=-1)
    grid_search.fit(X_train, y_train)

    model = grid_search.best_estimator_
    print("Best Parameters:", grid_search.best_params_)

    df['pred_proba'] = model.predict_proba(X)[:,1]
    return df, model

In [19]:
# === Step 3: Define Backtesting Strategy === #
class MLStrategy(Strategy):
    def init(self):
        self.probs = self.data.df['pred_proba'].values

    def next(self):
        idx = len(self.data.Close) - 1
        prob = self.probs[idx]

        if self.position:
            if (self.position.is_long and prob < 0.5) or (self.position.is_short and prob > 0.5):
                self.position.close()

        if prob > 0.55:
            fraction = min(1.0, 2 * (prob - 0.5))
            self.buy(size=fraction)
        elif prob < 0.45:
            fraction = min(1.0, 2 * (0.5 - prob))
            self.sell(size=fraction)

In [20]:
# === Step 4: Run Backtest === #
def run_backtest(df):
    df_bt = df.copy()
    df_bt.index = pd.to_datetime(df_bt.index)
    bt = Backtest(df_bt, MLStrategy, cash=100_000, commission=.002)
    stats = bt.run()
    bt.plot()
    return stats

In [21]:
df = pd.read_excel('eurusd-forex-data.xlsx')
df = add_features(df)
df, model = train_model(df)
stats = run_backtest(df)
print(stats)


Backtest.run:   0%|          | 0/3825 [00:00<?, ?bar/s]

Start                     1970-01-01 00:00...
End                       1970-01-01 00:00...
Duration                  0 days 00:00:00....
Exposure Time [%]                    96.91584
Equity Final [$]                7756528.99055
Equity Peak [$]                13000233.94344
Commissions [$]                19675528.84906
Return [%]                         7656.52899
Buy & Hold Return [%]               -10.64253
Return (Ann.) [%]                         0.0
Volatility (Ann.) [%]                     NaN
CAGR [%]                                  NaN
Sharpe Ratio                              NaN
Sortino Ratio                             NaN
Calmar Ratio                              0.0
Alpha [%]                          7657.59714
Beta                                  0.10037
Max. Drawdown [%]                   -43.79611
Avg. Drawdown [%]                    -0.56168
Max. Drawdown Duration    0 days 00:00:00....
Avg. Drawdown Duration    0 days 00:00:00....
# Trades                          

In [22]:
df2 = pd.read_excel('eurusd-forex-data.xlsx')
df2 = add_features(df)
df2, model = train_model_tuning(df)
stats = run_backtest(df)
print(stats)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Parameters: {'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 100, 'subsample': 1.0}


Backtest.run:   0%|          | 0/3825 [00:00<?, ?bar/s]

Start                     1970-01-01 00:00...
End                       1970-01-01 00:00...
Duration                  0 days 00:00:00....
Exposure Time [%]                    65.47308
Equity Final [$]                  513136.8759
Equity Peak [$]                  606119.05117
Commissions [$]                   247472.5455
Return [%]                          413.13688
Buy & Hold Return [%]               -10.64253
Return (Ann.) [%]                         0.0
Volatility (Ann.) [%]                     NaN
CAGR [%]                                  NaN
Sharpe Ratio                              NaN
Sortino Ratio                             NaN
Calmar Ratio                              0.0
Alpha [%]                           414.51146
Beta                                  0.12916
Max. Drawdown [%]                      -16.89
Avg. Drawdown [%]                     -0.3198
Max. Drawdown Duration    0 days 00:00:00....
Avg. Drawdown Duration    0 days 00:00:00....
# Trades                          

## Model Performance Comparison: Untuned vs Tuned

### Key Observations:

- **The hyperparameter-tuned model is more conservative.**  
  It likely avoided risky trades that the untuned model took, resulting in lower drawdowns.

- **Tuning parameters like `max_depth` and `learning_rate`**  
  leads to **reduced overfitting**, but can also make the model less aggressive — potentially missing some profitable trades.

- **Lower Final Equity in the tuned model**  
  is the trade-off for **improved risk management**.

- **Lower Maximum Drawdown**  
  makes the tuned strategy better suited for real-world deployment and institutional requirements.
